# RoadGuard: Comprehensive Experimental Evaluation (Fixed v10 - Hardened)
**Project Context**: Master's Thesis in Computer Science Engineering (Sapienza University of Rome)
**Dataset**: Thessaloniki & Larisa Road Quality Dataset
**Objective**: Optimized 100-epoch training and real-world evaluation with Drive backups.

---

## 1. Environment Setup
Mounting Drive and installing dependencies.

In [ ]:
!pip install ultralytics kaggle scikit-learn matplotlib pandas numpy tensorflow tf_keras -q
from google.colab import drive
import os, shutil, glob, zipfile, pickle, gc
from IPython.display import Image, display
import pandas as pd
import numpy as np

drive.mount('/content/drive')
DRIVE_SAVE_DIR = '/content/drive/MyDrive/RoadGuard_Thesis_Backups'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

if not os.path.exists('/content/RoadGuard'):
    !git clone https://github.com/antoninofoti/RoadGuard.git /content/RoadGuard
else:
    %cd /content/RoadGuard
    !git pull
    %cd /content

print('Environment and Drive Backup system ready.')

## 2. Dataset Acquisition & Incremental Synchronization
Processing real-world recordings with memory-efficient incremental persistence.

In [ ]:
from google.colab import files
if not os.path.exists('kaggle.json'):
    print("Action Required: Please upload your kaggle.json file")
    uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

THESSALONIKI_DIR = '/content/data/thessaloniki'
!mkdir -p {THESSALONIKI_DIR}
!kaggle datasets download nickkotarelas/road-quality-dataset -p {THESSALONIKI_DIR} --unzip

IMU_OUTPUT = os.path.join(THESSALONIKI_DIR, 'imu_normalized.csv')
VISION_OUTPUT = os.path.join(THESSALONIKI_DIR, 'frame_labels.csv')
if os.path.exists(IMU_OUTPUT): os.remove(IMU_OUTPUT)
if os.path.exists(VISION_OUTPUT): os.remove(VISION_OUTPUT)

pkl_files = glob.glob(f'{THESSALONIKI_DIR}/*.pkl')
for pkl_path in pkl_files:
    print(f"Processing session: {os.path.basename(pkl_path)}")
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    
    intervals = data.get('labels', [])
    
    # IMU
    imu = data.get('imu', {})
    accel, ts = imu.get('accel', []), imu.get('time', [])
    if len(accel) > 0:
        df = pd.DataFrame(accel, columns=['acc_x', 'acc_y', 'acc_z'])
        df['time'] = pd.to_numeric(ts, errors='coerce').astype(float)
        df['label'] = 0
        for l in intervals:
            s_t, e_t = float(l.get('rel_t_start', 0)), float(l.get('rel_t_end', 0))
            df.loc[(df['time'] >= s_t) & (df['time'] <= e_t), 'label'] = 1
        df.to_csv(IMU_OUTPUT, mode='a', index=False, header=not os.path.exists(IMU_OUTPUT))
        
    # Vision Metadata
    cam_times = data.get('camera', {}).get('time', [])
    if cam_times:
        v_meta = []
        for i, ct in enumerate(cam_times):
            curr, is_a = float(ct), 0
            for l in intervals:
                if float(l.get('rel_t_start', 0)) <= curr <= float(l.get('rel_t_end', 0)):
                    is_a = 1; break
            v_meta.append({'filename': f"frame_{i}.jpg", 'label': is_a})
        pd.DataFrame(v_meta).to_csv(VISION_OUTPUT, mode='a', index=False, header=not os.path.exists(VISION_OUTPUT))
    
    del data; gc.collect()

print('Real-world data synchronization complete.')

## 3. Vision Branch: 100-Epoch Training
Optimized training with automatic Drive backup.

In [ ]:
import os, shutil, glob, random, zipfile
!curl -L -o /content/pothole_dataset.zip 'https://ndownloader.figshare.com/files/37622126'
RAW_DIR = '/content/data/pothole_raw'
!mkdir -p {RAW_DIR}
if zipfile.is_zipfile('/content/pothole_dataset.zip'):
    !unzip -q /content/pothole_dataset.zip -d {RAW_DIR}
else:
    !kaggle datasets download -d vencerlanz09/pothole-segmentation-classification-detection -p /content/ --unzip
    RAW_DIR = '/content'

YOLO_DATA = '/content/data/pothole_yolo'
for s in ['train', 'val']: [os.makedirs(os.path.join(YOLO_DATA, f, s), exist_ok=True) for f in ['images', 'labels']]

def fix_and_copy_label(src, dst):
    with open(src, 'r') as f: lines = f.readlines()
    fixed = ["0" + line[line.find(" "):] for line in lines if line.strip()]
    with open(dst, 'w') as f: f.writelines(fixed)

all_imgs = []
for e in ['*.jpg', '*.jpeg', '*.png']: all_imgs.extend(glob.glob(os.path.join(RAW_DIR, '**', e), recursive=True))
pairs = [(i, os.path.splitext(i)[0] + '.txt') for i in all_imgs if os.path.exists(os.path.splitext(i)[0] + '.txt')]

if pairs:
    random.seed(42); random.shuffle(pairs); split = int(len(pairs) * 0.8)
    def move_and_fix(subset, sn):
        for img, lbl in subset:
            shutil.copy(img, os.path.join(YOLO_DATA, 'images', sn, os.path.basename(img)))
            fix_and_copy_label(lbl, os.path.join(YOLO_DATA, 'labels', sn, os.path.basename(lbl)))
    move_and_fix(pairs[:split], 'train'); move_and_fix(pairs[split:], 'val')

with open('/content/pothole_yolo.yaml', 'w') as f: f.write(f"path: {YOLO_DATA}\ntrain: images/train\nval: images/val\nnames:\n  0: pothole")

from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/pothole_yolo.yaml', epochs=100, imgsz=640, project='/content/runs', name='roadguard_v1')

# BACKUP TO DRIVE IMMEDIATELY
BEST_MODEL = '/content/runs/roadguard_v1/weights/best.pt'
if os.path.exists(BEST_MODEL):
    shutil.copy(BEST_MODEL, os.path.join(DRIVE_SAVE_DIR, 'best_100epochs.pt'))
    print(f"SUCCESS: Model backed up to Drive: {DRIVE_SAVE_DIR}/best_100epochs.pt")

model.export(format='tflite', int8=True, data='/content/pothole_yolo.yaml')
if os.path.exists('/content/runs/roadguard_v1/weights/best_int8.tflite'):
    shutil.copy('/content/runs/roadguard_v1/weights/best_int8.tflite', '/content/yolov8n_pothole.tflite')

## 4. Evaluation & FL (ABSOLUTE PATHS)
Formal validation on real synchronized recordings.

In [ ]:
%cd /content/RoadGuard/evaluation
IMU_CSV = "/content/data/thessaloniki/imu_normalized.csv"
V_MODEL = "/content/runs/roadguard_v1/weights/best.pt"
V_FRAMES = "/content/data/thessaloniki/frames"
V_LABELS = "/content/data/thessaloniki/frame_labels.csv"

!python eval_imu_branch.py --csv {IMU_CSV}
!python eval_vision_branch.py --model {V_MODEL} --frames {V_FRAMES} --labels {V_LABELS}
!python eval_late_fusion.py
!python generate_report.py
!python fl_partition.py
!python fl_personalized_fusion.py
%cd /content

## 5. Final Results Display & Archive
Packaging results for thesis integration.

In [ ]:
print('--- THESIS EVALUATION COMPLETE ---')
for img in ['/content/RoadGuard/evaluation/results/comparison_chart.png', 
            '/content/RoadGuard/evaluation/results/fl_dp_tradeoff.png',
            '/content/RoadGuard/evaluation/results/fl_full_comparison.png']:
    if os.path.exists(img): display(Image(img))

!mkdir -p /content/package/results
shutil.copytree('/content/RoadGuard/evaluation/results', '/content/package/results', dirs_exist_ok=True)
if os.path.exists('/content/yolov8n_pothole.tflite'):
    shutil.copy('/content/yolov8n_pothole.tflite', '/content/package/yolov8n_pothole.tflite')
shutil.make_archive('/content/RoadGuard_Final_Results', 'zip', '/content/package')
from google.colab import files
files.download('/content/RoadGuard_Final_Results.zip')